In [0]:
import logging
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

In [0]:

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("BronzeVeiculosEletricos")

def limpar_nulo_textual(col_name):
    """Converte strings vazias ou literais 'NULL'/'nan' em nulo real do Spark."""
    c = F.trim(F.col(col_name))
    return F.when((c == "") | (F.upper(c).isin("NULL", "NAN", "NONE")), None).otherwise(c)

In [0]:
logger.info("Processando bronze_emplacamentos_veiculos...")

df_bronze_emp = (
    spark.table("raw_emplacamentos_veiculos")
    .select(
        limpar_nulo_textual("mes").alias("ano_mes"),
        F.upper(limpar_nulo_textual("uf")).alias("uf"),
        limpar_nulo_textual("posicao").cast(IntegerType()).alias("posicao_ranking"),
        limpar_nulo_textual("marca").alias("marca"),
        limpar_nulo_textual("modelo").alias("modelo"),
        F.upper(limpar_nulo_textual("tecnologia")).alias("tecnologia"),
        limpar_nulo_textual("emplacamentos").cast(IntegerType()).alias("qtd_emplacamentos"),
        F.col("_data_ingestao")
    )
    .dropDuplicates()
)

df_bronze_emp.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_emplacamentos_veiculos")
logger.info(f"bronze_emplacamentos_veiculos gravada: {df_bronze_emp.count()} registros.")

In [0]:
logger.info("Processando bronze_ipva_veiculos_2026...")

df_bronze_ipva = (
    spark.table("raw_ipva_veiculos_2026")
    .select(
        F.upper(limpar_nulo_textual("uf")).alias("uf"),
        limpar_nulo_textual("estado").alias("nome_estado"),
        F.lower(limpar_nulo_textual("eletrico_regra")).alias("regra_ipva_eletrico"),
        limpar_nulo_textual("eletrico_aliquota_pct").cast(DoubleType()).alias("aliquota_eletrico_pct"),
        limpar_nulo_textual("eletrico_resumo").alias("resumo_ipva_eletrico"),
        F.lower(limpar_nulo_textual("hibrido_regra")).alias("regra_ipva_hibrido"),
        limpar_nulo_textual("hibrido_aliquota_pct").cast(DoubleType()).alias("aliquota_hibrido_pct"),
        limpar_nulo_textual("aliquota_geral_pct").cast(DoubleType()).alias("aliquota_geral_pct"),
        limpar_nulo_textual("confianca").alias("nivel_confianca"),
        F.col("_data_ingestao")
    )
    .dropDuplicates(["uf"])
)

df_bronze_ipva.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_ipva_veiculos_2026")
logger.info(f"bronze_ipva_veiculos_2026 gravada: {df_bronze_ipva.count()} registros.")

In [0]:
logger.info("Processando bronze_ficha_tecnica_veiculos...")

df_bronze_ficha = (
    spark.table("raw_ficha_tecnica_veiculos")
    .select(
        limpar_nulo_textual("id").alias("id_veiculo"),
        limpar_nulo_textual("marca").alias("marca"),
        limpar_nulo_textual("modelo").alias("modelo"),
        limpar_nulo_textual("versao").alias("versao"),
        F.upper(limpar_nulo_textual("motorizacao")).alias("tecnologia"),
        F.lower(limpar_nulo_textual("carroceria")).alias("carroceria"),
        F.lower(limpar_nulo_textual("condicao")).alias("condicao"),
        limpar_nulo_textual("preco_referencia_brl").cast(DoubleType()).alias("preco_referencia_brl"),
        limpar_nulo_textual("bateria_kwh").cast(DoubleType()).alias("bateria_kwh"),
        limpar_nulo_textual("autonomia_inmetro_km").cast(DoubleType()).alias("autonomia_inmetro_km"),
        limpar_nulo_textual("potencia_cv").cast(DoubleType()).alias("potencia_cv"),
        limpar_nulo_textual("recarga_dc_kw").cast(DoubleType()).alias("recarga_dc_kw"),
        limpar_nulo_textual("nota").cast(DoubleType()).alias("nota_avaliacao"),
        F.col("_data_ingestao")
    )
    .dropDuplicates(["id_veiculo"])
)

df_bronze_ficha.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_ficha_tecnica_veiculos")
logger.info(f"bronze_ficha_tecnica_veiculos gravada: {df_bronze_ficha.count()} registros.")

In [0]:
logger.info("Processando bronze_frota_senatran...")

df_bronze_senatran = (
    spark.table("raw_frota_senatran")
    .select(
        F.upper(limpar_nulo_textual("uf")).alias("estado_nome"),
        F.upper(limpar_nulo_textual("municipio")).alias("municipio"),
        F.upper(limpar_nulo_textual("combustivel")).alias("tipo_combustivel"),
        limpar_nulo_textual("qtd_veiculos").cast(IntegerType()).alias("qtd_veiculos"),
        F.col("_data_ingestao")
    )
    .filter(F.col("estado_nome").isNotNull() & F.col("qtd_veiculos").isNotNull())
)

df_bronze_senatran.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_frota_senatran")
logger.info(f"bronze_frota_senatran gravada: {df_bronze_senatran.count()} registros.")